# 🎓 SafeDrive AI: RAG Multimodal & Edge AI (Notebook Final TFG)

**Objetivo:** Arquitectura definitiva del sistema RAG local para el asistente post-accidente. 
Implementa Clean Code, ingesta multimodal con control de cuota (Gemini API), chunking optimizado para textos jurídico-médicos, y visualizaciones semánticas.

## 🗺️ Flujo del Sistema
1. Instalación de dependencias
2. Importaciones centralizadas (Clean Code)
3. Configuración Global (Hiperparámetros)
4. Ingesta Multimodal Segura (Gemini Vision + Rate Limiting)
5. Chunking de Alta Densidad Semántica
6. Indexación en ChromaDB (Local)
7. Pipeline RAG Edge (Inferencia Local)
8. Pruebas de Estrés
9. Visualización 2D (PCA Local)
10. Exportación a TensorFlow Embedding Projector

## 🔧 Paso 1: Instalación de dependencias

In [1]:
!pip install langchain langchain-chroma langchain-ollama chromadb pypdfium2 google-genai pillow scikit-learn matplotlib

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 📦 Paso 2: Importaciones Centralizadas (Clean Code)
Agrupamos todas las librerías en la primera celda ejecutable para evitar errores de dependencias en el flujo.

In [2]:
# 1. Librerías del Sistema y Datos
import os
import csv
import time
from pathlib import Path
from tqdm import tqdm

# 2. Librerías de Procesamiento de Imágenes y PDFs
from PIL import Image
import pypdfium2 as pdfium

# 3. Librería del Modelo de Visión 
import base64
from io import BytesIO
from langchain_core.messages import HumanMessage

# 4. Librerías de LangChain y Arquitectura RAG
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings, OllamaLLM, ChatOllama
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

# 5. Librerías para Análisis Matemático y Visualización (PCA)
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

print("✅ Todas las librerías importadas correctamente. Entorno listo.")

C:\Users\LittleDragon\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Todas las librerías importadas correctamente. Entorno listo.


## ⚙️ Paso 3: Configuración Global (Hiperparámetros y Rutas)
Definición estricta de parámetros. Hemos ajustado el `CHUNK_SIZE` a 512 y el `CHUNK_OVERLAP` a 150 para maximizar la cohesión de los artículos legales y evitar fragmentación semántica.

In [3]:
# --- RUTAS DE DIRECTORIOS ---
DATA_DIR = "../data/" 
CHROMA_DIR = "../chroma_db/dgt_multimodal"
COLLECTION_NAME = "BOE_dgt_seguridad"

# --- CONFIGURACIÓN DE MODELOS ---
OLLAMA_BASE_URL = "http://localhost:11434"
EMBEDDING_MODEL = "nomic-embed-text"
LLM_MODEL = "qwen2.5:7b"
OLLAMA_VISION_MODEL = "qwen2.5vl"

# --- CONFIGURACIÓN RAG (Densidad Semántica) ---
CHUNK_SIZE = 512
CHUNK_OVERLAP = 150

print(f"✅ Configuración global cargada: Chunk={CHUNK_SIZE}, Overlap={CHUNK_OVERLAP}")

✅ Configuración global cargada: Chunk=512, Overlap=150


## 📂 Paso 4: Ingesta Multimodal Segura (ETL)
Extrae texto y envía imágenes a Gemini para análisis técnico. Uso del modelo de visión local de Alibaba (Qwen2.5-VL)

In [ ]:
print(f"🤖 Preparando modelo de visión local: {OLLAMA_VISION_MODEL}...")
vision_llm = ChatOllama(model=OLLAMA_VISION_MODEL, base_url=OLLAMA_BASE_URL, temperature=0.0)

def procesar_pdf_multimodal_inteligente(ruta_pdf: str) -> list:
    print(f"\n📄 Procesando PDF: {Path(ruta_pdf).name}")
    documentos_generados = []
    pdf = pdfium.PdfDocument(ruta_pdf)
    
    for page_idx in range(len(pdf)):
        pagina = pdf[page_idx]
        
        # 1. Extraemos el texto nativo inmediatamente (Gratis y rápido)
        texto_nativo = pagina.get_textpage().get_text_bounded() or ""
        descripcion_grafica = "OMITIR"
        
        # 2. ESCÁNER GEOMÉTRICO INTELIGENTE (Filtro Sándwich Asimétrico)
        ancho_pag, alto_pag = pagina.get_size()
        tiene_grafico_real = False
        
        for obj in pagina.get_objects():
            if isinstance(obj, pdfium.PdfImage):
                left, bottom, right, top = obj.get_bounds()
                
                # REGLAS CALIBRADAS: 3 cm arriba (10%) y 2.3 cm abajo (8%)
                es_cabecera = bottom > (alto_pag * 0.90)  # 10% superior (franja azul/logo DGT arriba)
                es_pie_pagina = top < (alto_pag * 0.08)   # 8% inferior (logo DGT abajo/check verde)
                
                if es_cabecera or es_pie_pagina:
                    continue # Es basura corporativa repetitiva, la saltamos sin encender la IA
                
                # Si la imagen está en el cuerpo útil del documento... ¡Es un gráfico real!
                tiene_grafico_real = True
                break # Activamos la bandera y dejamos de escanear objetos en esta página
        
        # 3. Solo llamamos a la IA si realmente hay elementos visuales en el cuerpo
        if tiene_grafico_real:
            print(f"   📸 [Pág {page_idx + 1}] ¡Gráfico detectado en el cuerpo! Activando Qwen...")
            
            bitmap = pagina.render(scale=1.5)
            pil_img = bitmap.to_pil()
            
            buffered = BytesIO()
            pil_img.save(buffered, format="PNG")
            img_b64 = base64.b64encode(buffered.getvalue()).decode("utf-8")
            
            prompt_vision = (
                "Analiza los elementos visuales de esta página (señales, croquis, diagramas, tablas). "
                "Descríbelos detalladamente de forma técnica en español para un sistema de tráfico."
            )
            
            mensaje = HumanMessage(
                content=[
                    {"type": "text", "text": prompt_vision},
                    {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{img_b64}"}}
                ]
            )
            
            try:
                respuesta = vision_llm.invoke([mensaje])
                descripcion_grafica = respuesta.content
                print(f"   ✅ Análisis visual completado para pág {page_idx + 1}")
            except Exception as e:
                print(f"   ⚠️ Error analizando imagen en pág {page_idx + 1}: {e}")
        else:
            # Si solo había texto o logos en los márgenes, nos ahorramos por completo Ollama
            print(f"   🔤 [Pág {page_idx + 1}] Texto plano (márgenes DGT/BOE ignorados). Saltando visión.")
        
        # 4. Consolidamos el documento final para ChromaDB
        contenido_final = f"--- TEXTO DE LA PÁGINA ---\n{texto_nativo}\n"
        if "OMITIR" not in descripcion_grafica.upper() and descripcion_grafica.strip() != "":
            contenido_final += f"\n[ANÁLISIS MULTIMODAL DE GRÁFICOS Y SEÑALES]:\n{descripcion_grafica}"

        doc = Document(
            page_content=contenido_final,
            metadata={"source": Path(ruta_pdf).name, "pagina": page_idx + 1}
        )
        documentos_generados.append(doc)
        
    return documentos_generados

# Ejecución del pipeline
os.makedirs(DATA_DIR, exist_ok=True)
docs_totales = []
archivos_pdf = [f for f in os.listdir(DATA_DIR) if f.lower().endswith(".pdf")]

for archivo in archivos_pdf:
    ruta_completa = os.path.join(DATA_DIR, archivo)
    docs_totales.extend(procesar_pdf_multimodal_inteligente(ruta_completa))
    
print(f"\n✅ Carga Inteligente Sándwich completada. Páginas listas: {len(docs_totales)}")

🤖 Preparando modelo de visión local: qwen2.5vl...

📄 Procesando PDF: BOE-A-1995-25444-consolidado.pdf
   🔤 [Pág 1] Texto plano (márgenes DGT/BOE ignorados). Saltando visión.
   🔤 [Pág 2] Texto plano (márgenes DGT/BOE ignorados). Saltando visión.
   🔤 [Pág 3] Texto plano (márgenes DGT/BOE ignorados). Saltando visión.
   🔤 [Pág 4] Texto plano (márgenes DGT/BOE ignorados). Saltando visión.
   🔤 [Pág 5] Texto plano (márgenes DGT/BOE ignorados). Saltando visión.
   🔤 [Pág 6] Texto plano (márgenes DGT/BOE ignorados). Saltando visión.
   🔤 [Pág 7] Texto plano (márgenes DGT/BOE ignorados). Saltando visión.
   🔤 [Pág 8] Texto plano (márgenes DGT/BOE ignorados). Saltando visión.
   🔤 [Pág 9] Texto plano (márgenes DGT/BOE ignorados). Saltando visión.
   🔤 [Pág 10] Texto plano (márgenes DGT/BOE ignorados). Saltando visión.
   🔤 [Pág 11] Texto plano (márgenes DGT/BOE ignorados). Saltando visión.
   🔤 [Pág 12] Texto plano (márgenes DGT/BOE ignorados). Saltando visión.
   🔤 [Pág 13] Texto plano (márg

## ✂️ Paso 5: Chunking de Alta Densidad Semántica

In [8]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""]
)

splits = text_splitter.split_documents(docs_totales)

print(f"📊 Estadísticas del chunking:")
print(f"   Documentos originales: {len(docs_totales)}")
print(f"   Chunks generados:      {len(splits)}")
print(f"   Avg chars/chunk:       {sum(len(c.page_content) for c in splits) // len(splits)}")
print(f"   Max chars/chunk:       {max(len(c.page_content) for c in splits)}")
print(f"   Min chars/chunk:       {min(len(c.page_content) for c in splits)}")

print(f"✂️ Documentos divididos con éxito en {len(splits)} fragmentos (chunks) vectorizables.")

📊 Estadísticas del chunking:
   Documentos originales: 0
   Chunks generados:      0


ZeroDivisionError: integer division or modulo by zero

## 🗄️ Paso 6: Indexación en ChromaDB Local

In [ ]:
embeddings = OllamaEmbeddings(model=EMBEDDING_MODEL, base_url=OLLAMA_BASE_URL)

print("Guardando vectores en base de datos local...")
vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embeddings,
    collection_name=COLLECTION_NAME,
    persist_directory=CHROMA_DIR,
    collection_metadata={"hnsw:space": "cosine"}
)

print(f"✅ ChromaDB persistido en: {CHROMA_DIR}")

## 🧠 Paso 7: Pipeline RAG Edge (Inferencia Local)

In [ ]:
# Inicializar LLM local con Ollama
print(f"🤖 Cargando LLM: {LLM_MODEL}")

retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
llm = OllamaLLM(
    model=LLM_MODEL, 
    base_url=OLLAMA_BASE_URL,
    temperature=0.1,
    num_ctx=4096,
)

system_prompt = """Eres SafeDrive AI, un asistente experto en seguridad vial, primeros auxilios y normativa de la DGT (Dirección General de Tráfico de España).
Tu objetivo es asistir a conductores que acaban de sufrir un accidente de tráfico.
Responde de forma clara, directa y empática, priorizando siempre la seguridad física y la protección legal del usuario.

Usa ÚNICAMENTE los siguientes fragmentos de contexto recuperado de los manuales oficiales de la DGT para responder a la pregunta.
Si la respuesta no está en el contexto o tienes dudas, di estrictamente: "Por seguridad, no puedo ofrecer asesoramiento sobre ese tema. Por favor, contacta con el 112 si es una emergencia médica."
NO inventes leyes, NO alucines procedimientos médicos y NO des consejos que contradigan el contexto proporcionado.

Contexto recuperado:
{context}

Pregunta del conductor: {question}

Respuesta de SafeDrive AI:"""

PROMPT = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"]
)

print("✅ LLM y prompt configurados")


In [ ]:
# Formateador formal del contexto para auditoría visual por consola
def format_docs(docs):
    return "\n\n".join([f"[Origen: {d.metadata['source']} | Pág: {d.metadata['pagina']}]\n{d.page_content}" for d in docs])

# Construcción de la cadena LCEL
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("🚀 Pipeline de inferencia RAG Edge listo.")

## 📋 Paso 8: Pruebas de Estrés

In [ ]:
def consultar_asistente(pregunta: str):
    print(f"\n🙋‍♂️ Conductor: {pregunta}")
    print("-" * 60)
    
    # Simulación previa del recuperador para la auditoría técnica por consola
    docs_recuperados = retriever.invoke(pregunta)
    print(f"🔍 Chunks recuperados matemáticamente: {len(docs_recuperados)}")
    for d in docs_recuperados:
        print(f"   - Fuente: {d.metadata['source']} (Pág. {d.metadata['pagina']})")
        
    # Inferencia de la cadena integrada
    respuesta = rag_chain.invoke(pregunta)
    print(f"\n🤖 SafeDrive AI:\n{respuesta}\n")
    print("=" * 80)

# --- EJECUCIÓN DE CASOS DE CONTROL DEL TFG ---
consultar_asistente("El otro conductor quiere que firme el parte en blanco para rellenarlo luego, ¿debo hacerlo?")
consultar_asistente("Tengo una hemorragia en el brazo tras el choque, ¿qué hago?")
consultar_asistente("He chocado en una curva donde había una señal triangular blanca con borde rojo. ¿Cómo pongo el vehículo de auxilio?")

## 🎨 Paso 9: Visualización 2D (PCA Local)
Proyecta los vectores matemáticos generados por Nomic en un plano 2D para comprobar visualmente si el modelo ha agrupado los documentos por conceptos.

In [ ]:
print("🎨 Extrayendo vectores y generando proyección PCA 2D...")

# 1. Extraemos todo el contenido de la colección para este paso y el siguiente
datos_coleccion = vectorstore.get(include=["embeddings", "metadatas", "documents"])
vectores = np.array(datos_coleccion["embeddings"])

# 2. Extraemos las fuentes para asignar colores
fuentes = [meta.get("source", "Desconocido") for meta in datos_coleccion["metadatas"]]
fuentes_unicas = list(set(fuentes))

# 3. Reducción dimensional PCA (De >700 dimensiones a 2)
pca = PCA(n_components=2)
coords_2d = pca.fit_transform(vectores)

# 4. Renderizado del gráfico
plt.figure(figsize=(12, 8))
colores = plt.cm.get_cmap('tab10', len(fuentes_unicas))

for i, fuente in enumerate(fuentes_unicas):
    idx = [j for j, f in enumerate(fuentes) if f == fuente]
    plt.scatter(
        coords_2d[idx, 0], coords_2d[idx, 1], 
        alpha=0.7, label=fuente, color=colores(i),
        edgecolors='w', linewidth=0.5, s=60
    )

plt.title("Espacio Semántico de SafeDrive AI (PCA)", fontsize=14, pad=15)
plt.xlabel("Componente Principal 1", fontsize=11)
plt.ylabel("Componente Principal 2", fontsize=11)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
plt.grid(True, alpha=0.3, linestyle='--')
plt.tight_layout()

plt.show()

## 📊 Paso 10: Exportar a TensorFlow Embedding Projector
Genera los archivos `.tsv` para navegar el espacio tridimensional en la web oficial de TensorFlow.

In [ ]:
print("📦 Exportando base de datos a formato TSV...")

# Exportar embeddings.tsv
with open("embeddings.tsv", "w", newline="", encoding="utf-8") as f_vec:
    writer = csv.writer(f_vec, delimiter="\t")
    writer.writerows(datos_coleccion["embeddings"])

# Exportar metadata.tsv
with open("metadata.tsv", "w", newline="", encoding="utf-8") as f_meta:
    writer = csv.writer(f_meta, delimiter="\t")
    writer.writerow(["Fuente", "Pagina", "Texto_Muestra"]) 
    
    for meta, doc in zip(datos_coleccion["metadatas"], datos_coleccion["documents"]):
        texto_limpio = doc.replace("\n", " ").replace("\t", " ")[:150] + "..."
        writer.writerow([meta.get("source", "Desconocido"), meta.get("pagina", 0), texto_limpio])

print("✅ ¡Archivos 'embeddings.tsv' y 'metadata.tsv' generados con éxito!")
print("👉 Sube ambos archivos a: https://projector.tensorflow.org/")